<a href="https://colab.research.google.com/github/ArjiNasim/mrm-oversight-engine/blob/main/mrm_oversight_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ==========================================
# CONFIGURATION DE LA PAGE
# ==========================================
st.set_page_config(
    page_title="MRM Oversight Dashboard",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Style CSS personnalisé pour un look bancaire/institutionnel
st.markdown("""
    <style>
    .metric-container {
        background-color: #f8f9fa;
        border-radius: 5px;
        padding: 15px;
        border-left: 4px solid #004085;
    }
    .metric-red { border-left: 4px solid #dc3545; }
    .metric-green { border-left: 4px solid #28a745; }
    </style>
""", unsafe_allow_html=True)

# ==========================================
# 2. SIMULATION DES DONNÉES DE PERFORMANCE
# ==========================================
@st.cache_data
def load_mock_data():
    np.random.seed(42)
    models = [
        "PD IFRS 9 - Corporate", "PD IFRS 9 - Retail", "LGD IFRS 9 - Mortgages",
        "Scoring Octroi Particuliers", "Notation Interne Entreprises",
        "ALM - Taux d'intérêt", "Market Risk - VaR Historique", "AML - Transaction Monitoring"
    ]

    n_models = 45 # Total modèles dans le portefeuille

    df_portfolio = pd.DataFrame({
        "ID_Modele": [f"MOD-{i:03d}" for i in range(1, n_models + 1)],
        "Nom_Modele": np.random.choice(models, n_models),
        "Type": np.random.choice(["Credit Risk", "Market Risk", "ALM", "Compliance (AML)"], n_models, p=[0.5, 0.2, 0.15, 0.15]),
        "Tier": np.random.choice(["Tier 1 (High)", "Tier 2 (Medium)", "Tier 3 (Low)"], n_models, p=[0.3, 0.5, 0.2]),
        "Statut": np.random.choice(["Validé", "Sous Surveillance", "En Cours de Validation", "Rejeté"], n_models, p=[0.6, 0.2, 0.15, 0.05]),
        "PSI": np.random.uniform(0.01, 0.35, n_models),
        "AUC": np.random.uniform(0.55, 0.95, n_models)
    })

    df_portfolio.loc[df_portfolio['Statut'] == 'Sous Surveillance', 'PSI'] += 0.15
    df_portfolio.loc[df_portfolio['Statut'] == 'Rejeté', 'PSI'] += 0.20
    df_portfolio.loc[df_portfolio['Statut'] == 'Rejeté', 'AUC'] -= 0.15

    return df_portfolio

@st.cache_data
def load_timeseries_data(model_name):
    dates = [datetime.today() - timedelta(days=x*30) for x in range(12)]
    dates.reverse()

    base_auc = 0.85
    base_psi = 0.10

    drift_auc = np.linspace(0, -0.12, 12) + np.random.normal(0, 0.01, 12)
    drift_psi = np.linspace(0, 0.18, 12) + np.random.normal(0, 0.02, 12)

    df_ts = pd.DataFrame({
        "Date": dates,
        "AUC": base_auc + drift_auc,
        "PSI": base_psi + drift_psi
    })
    return df_ts

df = load_mock_data()

# ==========================================
# 1. SECTION DE SYNTHÈSE (KPIs Clés)
# ==========================================
st.title("🛡️ MRM Oversight Dashboard")
st.markdown("##### 2nd Line of Defence - Model Risk Management")
st.divider()

col1, col2, col3, col4 = st.columns(4)

total_models = len(df)
red_alerts = len(df[(df['PSI'] > 0.25) | (df['AUC'] < 0.65)])
compliance_rate = len(df[df['Statut'] == 'Validé']) / total_models * 100
avg_psi = df['PSI'].mean()

with col1:
    st.metric("Total Modèles Supervisés", total_models)
with col2:
    st.metric("Alertes Rouges (Performance)", red_alerts, delta="-2 depuis le mois dernier", delta_color="inverse")
with col3:
    st.metric("Taux de Conformité", f"{compliance_rate:.1f}%")
with col4:
    st.metric("PSI Moyen du Portefeuille", f"{avg_psi:.3f}")

st.divider()

Overwriting app.py


In [5]:
%%writefile -a app.py

import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 3. VISUALISATIONS INTERACTIVES AVEC PLOTLY
# ==========================================
col_chart1, col_chart2 = st.columns(2)

with col_chart1:
    st.subheader("Analyse de la Stabilité (PSI) par Modèle")

    # Création d'une colonne de couleur conditionnelle pour le PSI
    df['Alerte_PSI'] = np.where(df['PSI'] > 0.25, 'Critique (>0.25)',
                       np.where(df['PSI'] > 0.10, 'Alerte (0.10 - 0.25)', 'Stable (<0.10)'))

    color_discrete_map = {'Critique (>0.25)': '#dc3545', 'Alerte (0.10 - 0.25)': '#ffc107', 'Stable (<0.10)': '#28a745'}

    fig_psi = px.scatter(df, x="AUC", y="PSI", color="Alerte_PSI",
                         hover_name="ID_Modele", hover_data=["Nom_Modele", "Type", "Statut"],
                         color_discrete_map=color_discrete_map,
                         title="Matrice de Risque : Pouvoir Discriminant (AUC) vs Stabilité (PSI)")

    # Lignes de seuils
    fig_psi.add_hline(y=0.25, line_dash="dash", line_color="red", annotation_text="Seuil Critique PSI (0.25)")
    fig_psi.add_vline(x=0.65, line_dash="dash", line_color="red", annotation_text="Seuil Critique AUC (0.65)")
    fig_psi.update_layout(height=450, template="plotly_white")
    st.plotly_chart(fig_psi, use_container_width=True)

with col_chart2:
    st.subheader("Suivi Temporel de la Performance")
    selected_model = st.selectbox("Sélectionner un modèle à inspecter :", df[df['Statut'] == 'Sous Surveillance']['ID_Modele'].tolist())

    if selected_model:
        model_details = df[df['ID_Modele'] == selected_model].iloc[0]
        st.markdown(f"**Modèle :** {model_details['Nom_Modele']} | **Tier :** {model_details['Tier']}")

        df_ts = load_timeseries_data(selected_model)

        fig_ts = go.Figure()
        fig_ts.add_trace(go.Scatter(x=df_ts['Date'], y=df_ts['AUC'], mode='lines+markers', name='AUC (Performance)', line=dict(color='#004085', width=3)))
        fig_ts.add_trace(go.Scatter(x=df_ts['Date'], y=df_ts['PSI'], mode='lines+markers', name='PSI (Instabilité)', yaxis='y2', line=dict(color='#dc3545', width=3, dash='dot')))

        fig_ts.update_layout(
            title="Évolution sur 12 mois (Drift Analysis)",
            yaxis=dict(title='AUC', range=[0.5, 1]),
            yaxis2=dict(title='PSI', overlaying='y', side='right', range=[0, 0.4]),
            height=370, template="plotly_white",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        st.plotly_chart(fig_ts, use_container_width=True)

st.divider()

# ==========================================
# 4. SECTION "IA & NLP ASSISTANT"
# ==========================================
st.subheader("🤖 Assistant IA & NLP (Analyse des Rapports de Validation)")
st.markdown("Ce module utilise un LLM pour extraire les faiblesses méthodologiques (Findings) et synthétiser les avis du comité de validation.")

nlp_col1, nlp_col2 = st.columns([1, 2])

with nlp_col1:
    report_text = st.text_area("Extrait du rapport de validation (IFRS 9 / IRB) :",
                               "The model demonstrates a material degradation in the discriminatory power for the Retail mortgage portfolio. Specifically, the Gini coefficient dropped from 42% to 28% post-COVID. Furthermore, the macroeconomic overlays applied lack sufficient empirical justification and rely heavily on expert judgment. The LGD component is sensitive to collateral valuation shocks.",
                               height=200)
    analyze_btn = st.button("Lancer l'analyse NLP", type="primary")

with nlp_col2:
    if analyze_btn:
        import time
        with st.spinner("Traitement NLP en cours (Extraction des entités de risque)..."):
            time.sleep(1.5)
            st.success("Analyse terminée.")
            st.markdown("#### Synthèse Exécutive du Modèle (Générée par IA)")
            st.markdown('''
            * **🔥 Alerte Majeure :** Baisse significative du pouvoir discriminant (Gini -14 points) sur le portefeuille Retail.
            * **⚠️ Risque Méthodologique :** Dépendance excessive au jugement d'expert pour les *forward-looking overlays* (IFRS 9). Manque de backtesting empirique.
            * **📉 Sensibilité LGD :** Vulnérabilité identifiée face aux chocs de valorisation des garanties immobilières (collateral).

            **Recommandation MRM automatique :**
            `ACTION REQUISE` - Exiger de la 1ère Ligne de Défense (1LoD) une recalibration du modèle Retail et une documentation détaillée des scénarios macroéconomiques sous 60 jours.
            ''')
    else:
        st.info("Cliquez sur 'Lancer l'analyse NLP' pour extraire les clauses de risque réglementaire et résumer le rapport.")

Appending to app.py
